> **Gabarito.** Abra só depois de tentar resolver a versão do aluno.

# Aula 7 – PCA exploratória e correlação

**Objetivo:** resumir milhares de bandas em poucos componentes e descobrir quais atributos do solo cada componente representa.

## Preparação
Rode esta célula primeiro. Ela importa as funções das aulas anteriores.

In [ ]:
import sys
from pathlib import Path
# Encontra a pasta aulas/ (onde está curso_utils.py), rodando de aulas/ ou de aulas/solucoes/
_aqui = Path.cwd()
_aulas = _aqui if (_aqui / 'curso_utils.py').exists() else _aqui.parent
sys.path.insert(0, str(_aulas))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from curso_utils import preparar_nir, ALVOS
meta, X, wl = preparar_nir(n=500)
print(X.shape)

## Exercício 1 – PCA por SVD
Devolva `(scores, loadings, var)`, onde `var` é a **fração** da variância total de cada PC.

In [ ]:
def pca_svd(X, k):
    Xc = X - X.mean(0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    var = S ** 2 / np.sum(S ** 2)
    return U[:, :k] * S[:k], Vt[:k], var[:k]

scores, loadings, var = pca_svd(X, 8)
print(np.round(var * 100, 1), '% | soma:', round(var.sum() * 100, 1))

## Exercício 2 – Correlação dos PCs com os atributos
Devolva um DataFrame com alvos nas linhas e PCs nas colunas. Ignore amostras com o alvo faltando.

In [ ]:
def correlacoes(scores, meta, alvos):
    linhas = {}
    for alvo in alvos:
        if alvo not in meta.columns:
            continue
        y = pd.to_numeric(meta[alvo], errors='coerce').to_numpy(float)
        ok = np.isfinite(y)
        linhas[alvo] = [np.corrcoef(scores[ok, j], y[ok])[0, 1] for j in range(scores.shape[1])]
    return pd.DataFrame(linhas, index=[f'PC{j + 1}' for j in range(scores.shape[1])]).T

R = correlacoes(scores, meta, ALVOS)
R.round(2)

## Exercício 3 – Heatmap e loadings

In [ ]:
plt.imshow(R.to_numpy(), cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(label='r de Pearson')
plt.yticks(range(len(R)), R.index); plt.xticks(range(R.shape[1]), R.columns)
plt.title('Correlação PCs × atributos'); plt.show()

j = int(np.argmax(np.abs(R.loc['Clay_gkg'].to_numpy())))
plt.plot(wl, loadings[j])
for x in (1400, 1900, 2200):
    plt.axvline(x, color='purple', ls=':')
plt.title(f'Loadings do PC{j + 1} (mais correlacionado com argila)'); plt.xlabel('nm'); plt.show()

In [ ]:
# PC1 x PC2 colorido pelo teor de argila
plt.scatter(scores[:, 0], scores[:, 1], c=pd.to_numeric(meta['Clay_gkg'], errors='coerce'), s=10, cmap='viridis')
plt.colorbar(label='Argila (g/kg)'); plt.xlabel('PC1'); plt.ylabel('PC2'); plt.show()

## Checkpoint

Por que r = −0,8 é tão informativo quanto r = +0,8? O sinal de um PC tem significado físico?